In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


def choose_device():
    # 选择当前 Kaggle 环境里真正能安全使用的设备。
    if not torch.cuda.is_available():
        print('GPU not found, using CPU. 也能跑小实验，但会慢一些。')
        return 'cpu'

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print('GPU:', name)
    print('CUDA capability:', f'sm_{props.major}{props.minor}')

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 如果当前 PyTorch 构建不支持 sm_60，torch.cuda.is_available() 仍可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print()
        print('注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建可能不支持 sm_60。')
        print('本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。')
        print('如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。')
        print()
        return 'cpu'

    return 'cuda'


device = choose_device()


# 固定随机种子。
# 初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == 'cuda':
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对学习实验来说，可以让矩阵乘法更快。
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print('Using device:', device)
print('Seed:', seed)


# 后续我们会按这个顺序继续写：
# Cell 2：按官方 GPT-2 复现路线准备 OpenWebText，并用 GPT-2 BPE 变成 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：实现最小语言模型，先理解训练闭环
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本


In [ ]:
# 第 2 个 cell：官方 OpenWebText + GPT-2 BPE 数据准备
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/openwebtext/prepare.py
#
# 你提到的“基于 openwebtext dataset 复现 GPT-2”，对应的就是这条路线：
#
#   OpenWebText -> GPT-2 BPE tokenizer -> train.bin / val.bin
#
# 注意：完整 OpenWebText 很大。nanoGPT 官方注释里写到：
#   Hugging Face cache 约 54GB
#   train.bin 约 17GB
#   val.bin 约 8.5MB
#   train 约 9B tokens
#
# 所以这一格优先使用 Kaggle Input 里已经准备好的 train.bin / val.bin。
# 如果你确认 Kaggle 磁盘和时间足够，再把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，
# 它会按官方 prepare.py 的方式从 Hugging Face 下载并预处理完整 OpenWebText。

import os
import subprocess
import sys
from pathlib import Path

import numpy as np


PREPARE_OPENWEBTEXT_FROM_HF = False


def ensure_package(import_name, pip_name=None):
    # 缺少依赖时自动安装，方便在 Kaggle 新环境中直接运行。
    try:
        return __import__(import_name)
    except ModuleNotFoundError:
        package_name = pip_name or import_name
        print(f'当前环境没有 {package_name}，正在安装。')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        return __import__(import_name)


tiktoken = ensure_package('tiktoken')
ensure_package('tqdm')
from tqdm import tqdm


# GPT-2 BPE tokenizer。
# BPE 可以理解成“常见文本片段优先合并”的分词方式。
# GPT-2 的普通词表大小是 50257，其中结束符通常是 50256。
# nanoGPT 从零训练 GPT-2 规模模型时，默认把 vocab_size padding 到 50304，
# 这样矩阵维度更适合 GPU 计算。
enc = tiktoken.get_encoding('gpt2')
gpt2_vocab_size = enc.n_vocab
vocab_size = 50304

print('Tokenizer:', enc.name)
print('GPT-2 原始词表大小:', gpt2_vocab_size)
print('GPT-2 eot_token:', enc.eot_token)
print('nanoGPT 从零训练默认 vocab_size:', vocab_size)


example_text = 'The quick brown fox jumps over the lazy dog.'
example_ids = enc.encode_ordinary(example_text)

print()
print('BPE 编码/解码小例子：')
print('原始文本:', example_text)
print('编码后 token id:', example_ids)
print('解码回去:', enc.decode(example_ids))
print('每个 token id 对应的文本片段：')
for token_id in example_ids:
    print(f'{token_id:>6} -> {enc.decode([token_id])!r}')


def find_prepared_openwebtext_bins():
    # 优先在 Kaggle Input 里找已经准备好的 OpenWebText 二进制 token 文件。
    # 这样不占 /kaggle/working 的 Output 空间。
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists():
        return None

    train_candidates = sorted(kaggle_input.rglob('train.bin'))
    val_candidates = sorted(kaggle_input.rglob('val.bin'))

    for train_path in train_candidates:
        same_dir_val = train_path.with_name('val.bin')
        if same_dir_val.exists():
            return train_path, same_dir_val

    if train_candidates and val_candidates:
        return train_candidates[0], val_candidates[0]

    return None


def prepare_openwebtext_from_hf():
    # 这部分严格对应 nanoGPT 官方 data/openwebtext/prepare.py。
    # 完整运行会下载和处理大量数据，Kaggle 免费环境可能因为空间或时间失败。
    ensure_package('datasets')
    from datasets import load_dataset

    if Path('/kaggle/working').exists():
        data_dir = Path('/kaggle/working/data/openwebtext')
    else:
        data_dir = Path('data/openwebtext')
    data_dir.mkdir(parents=True, exist_ok=True)

    num_proc = min(8, max(1, (os.cpu_count() or 2) // 2))
    num_proc_load_dataset = num_proc

    print('开始加载 OpenWebText。第一次运行会下载大量数据。')
    dataset = load_dataset('openwebtext', num_proc=num_proc_load_dataset)

    split_dataset = dataset['train'].train_test_split(
        test_size=0.0005,
        seed=2357,
        shuffle=True,
    )
    split_dataset['val'] = split_dataset.pop('test')
    print(split_dataset)

    def process(example):
        ids = enc.encode_ordinary(example['text'])
        ids.append(enc.eot_token)
        return {'ids': ids, 'len': len(ids)}

    print('开始 GPT-2 BPE 分词。')
    tokenized = split_dataset.map(
        process,
        remove_columns=['text'],
        desc='tokenizing the splits',
        num_proc=num_proc,
    )

    bin_paths = {}
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = data_dir / f'{split}.bin'
        arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
        total_batches = 1024
        idx = 0

        print()
        print(f'开始写入 {filename}')
        print(f'{split} token 总数:', f'{int(arr_len):,}')

        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename.name}'):
            batch = dset.shard(
                num_shards=total_batches,
                index=batch_idx,
                contiguous=True,
            ).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()
        bin_paths[split] = filename

    return bin_paths['train'], bin_paths['val']


prepared_bins = find_prepared_openwebtext_bins()

if prepared_bins is not None:
    train_bin_path, val_bin_path = prepared_bins
    print()
    print('发现 Kaggle Input 中已有 OpenWebText bin 文件，直接读取：')
    print('train.bin:', train_bin_path)
    print('val.bin:', val_bin_path)
elif PREPARE_OPENWEBTEXT_FROM_HF:
    train_bin_path, val_bin_path = prepare_openwebtext_from_hf()
else:
    raise FileNotFoundError(
        '没有在 /kaggle/input 中找到 train.bin / val.bin。\n'
        '推荐先在 Kaggle Add Input 里添加已经准备好的 OpenWebText 数据集。\n'
        '如果你确认空间和时间足够，也可以把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，按官方脚本现场下载预处理。'
    )


# 后面训练时不需要把完整 train.bin 读进内存。
# np.memmap 会建立“文件视图”，用到哪一段再读哪一段。
train_data = np.memmap(train_bin_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode='r')

print()
print('OpenWebText 数据准备完成。')
print('train.bin:', train_bin_path, 'tokens:', f'{len(train_data):,}')
print('val.bin:', val_bin_path, 'tokens:', f'{len(val_data):,}')
print('训练集前 40 个 token id:')
print(train_data[:40].tolist())
print('解码前 40 个 token:')
print(enc.decode(train_data[:40].tolist()))


def encode(s):
    # 把字符串变成 GPT-2 BPE token id 列表。
    return enc.encode_ordinary(s)


def decode(ids):
    # 把 GPT-2 BPE token id 列表还原成字符串。
    return enc.decode(ids)


# 这一格产生的关键变量：
#   enc / encode / decode       GPT-2 BPE 分词器和辅助函数
#   vocab_size                  nanoGPT 从零训练 GPT-2 路线默认词表大小，50304
#   train_bin_path / val_bin_path
#   train_data / val_data       np.memmap 形式的 OpenWebText token 序列
#
# 下一格 Cell 3 会讲：
#   如何从 train_data 中随机取一批连续 token，
#   构造 x 和 y，让模型学习“看到前文，预测下一个 token”。


In [ ]:
# 第 3 个 cell：实现 get_batch，构造语言模型的 x 和 y
#
# 这一格对应 nanoGPT 官方源码 train.py 里的 get_batch(split)。
#
# 语言模型训练的核心任务非常朴素：
#   给模型看一段前文 x，让它预测每个位置的下一个 token，也就是 y。
#
# 假设原始 token 序列是：
#   [10, 20, 30, 40, 50]
# 如果 block_size = 4，那么：
#   x = [10, 20, 30, 40]
#   y = [20, 30, 40, 50]
#
# 这不是随便错开一位，而是在告诉模型：
#   看到 10，应该预测 20
#   看到 10,20，应该预测 30
#   看到 10,20,30，应该预测 40
#   看到 10,20,30,40，应该预测 50
#
# 后面算 loss 的时候，就是把模型每个位置预测出来的概率分布，
# 和这里的 y 逐个对答案。


# GPT-2 的默认上下文长度是 1024。
# 也就是说，模型每次最多看前面 1024 个 token 来预测后面的 token。
# 这里 batch_size 先设小一点，方便在 Kaggle 上观察输出，后面训练时可以再调大。
block_size = 1024
batch_size = 4


def get_batch(split):
    # split 决定从训练集还是验证集取样。
    # train_data / val_data 是上一格得到的 np.memmap，里面是一长串 GPT-2 BPE token id。
    data = train_data if split == 'train' else val_data

    if len(data) <= block_size:
        raise ValueError(
            f'{split} 数据太短：只有 {len(data)} 个 token，'
            f'但 block_size={block_size}。请换更大的数据，或减小 block_size。'
        )

    # ix 是一批随机起点。
    # 每个起点 i 都会切出一段长度为 block_size 的 x，
    # 再切出一段向右平移一位的 y。
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x_np = np.stack([
        np.array(data[int(i) : int(i) + block_size], dtype=np.int64)
        for i in ix
    ])
    y_np = np.stack([
        np.array(data[int(i) + 1 : int(i) + 1 + block_size], dtype=np.int64)
        for i in ix
    ])

    x = torch.from_numpy(x_np)
    y = torch.from_numpy(y_np)

    # 如果真能用 CUDA，就把 batch 搬到 GPU。
    # pin_memory + non_blocking 是官方 train.py 里的加速写法：
    # 它让 CPU 到 GPU 的数据搬运更顺畅。
    if device == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x = x.to(device)
        y = y.to(device)

    return x, y, ix


xb, yb, ix = get_batch('train')

print('随机起点 ix:', ix.tolist())
print('x shape:', tuple(xb.shape))
print('y shape:', tuple(yb.shape))
print('x dtype/device:', xb.dtype, xb.device)
print('y dtype/device:', yb.dtype, yb.device)


# 看第一条样本的前 40 个 token。
# x 和 y 应该刚好错开一位。
sample_row = 0
show_tokens = 40

x_head = xb[sample_row, :show_tokens].detach().cpu().tolist()
y_head = yb[sample_row, :show_tokens].detach().cpu().tolist()

print()
print('第一条样本 x 的前 40 个 token id:')
print(x_head)
print('第一条样本 y 的前 40 个 token id:')
print(y_head)

print()
print('把 x 前 40 个 token 解码成文字：')
print(decode(x_head))
print('把 y 前 40 个 token 解码成文字：')
print(decode(y_head))


# 更细一点看：每个位置到底在学什么？
# 下面打印前 8 个位置。
# 左边是模型能看到的上下文，右边是它该预测的下一个 token。
print()
print('前 8 个训练小任务：')
for t in range(8):
    context_ids = xb[sample_row, : t + 1].detach().cpu().tolist()
    target_id = int(yb[sample_row, t].detach().cpu().item())
    context_text = decode(context_ids)
    target_text = decode([target_id])
    print(f'看到 {context_text!r} -> 预测下一个 token {target_text!r}  token_id={target_id}')


# 这一格产生的关键变量：
#   block_size      每条样本的上下文长度，GPT-2 默认是 1024
#   batch_size      每次并行取多少条样本
#   get_batch       从 train_data / val_data 随机取 x/y 的函数
#   xb / yb         一个实际训练 batch，可以直接喂给后面的模型
#
# 下一格 Cell 4 会先实现一个最小语言模型，
# 看看 logits、loss、反向传播和优化器到底如何连起来。


In [ ]:
# 第 4 个 cell：直接写入 nanoGPT 官方 model.py 的 GPT 模型定义
#
# 这一格不再使用“最小模型”或省显存版本。
# 下面的 LayerNorm、CausalSelfAttention、MLP、Block、GPTConfig、GPT 都来自 nanoGPT 官方 model.py。
# 后续训练、优化器、生成文本都会围绕这个 GPT 类继续写。
#
# 对应源码：nanoGPT-reference/model.py

"""
Full definition of a GPT Language Model, all of it in this single file.
References:
1) the official GPT-2 TensorFlow implementation released by OpenAI:
https://github.com/openai/gpt-2/blob/master/src/model.py
2) huggingface/transformers PyTorch implementation:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
"""

import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# 用官方 GPT-2 124M 默认配置实例化一次，确认模型结构和 batch 能接上。
# 这里不做训练，只做很小的 forward smoke test。
# 官方从零训练 GPT-2 124M 的默认关键参数：
#   block_size=1024, vocab_size=50304, n_layer=12, n_head=12, n_embd=768

model_args = dict(
    n_layer=12,
    n_head=12,
    n_embd=768,
    block_size=block_size,
    bias=True,
    vocab_size=vocab_size,
    dropout=0.0,
)

gpt_config = GPTConfig(**model_args)
model = GPT(gpt_config).to(device)
model.eval()

print('GPTConfig:', gpt_config)
print('模型参数量:', f'{model.get_num_params() / 1e6:.2f}M')
print('当前 device:', device)

# 只取很短的一段做形状检查，避免在这格里做完整训练。
debug_x = xb[:1, :8].to(device)
debug_y = yb[:1, :8].to(device)

with torch.no_grad():
    debug_logits, debug_loss = model(debug_x, debug_y)

print('debug_x shape:', tuple(debug_x.shape))
print('debug_logits shape:', tuple(debug_logits.shape))
print('debug_loss:', float(debug_loss.item()))

# 这一格产生的关键变量：
#   LayerNorm / CausalSelfAttention / MLP / Block / GPTConfig / GPT
#   gpt_config
#   model
#
# 下一格 Cell 5 会按 nanoGPT train.py 来设置 optimizer、学习率和一次训练 step。
